# YOLO Adversarial Challenge Solver
**Goal:** Generate an image that is visually similar to `source_image.jpg` ($L_2$ distance < 50) but fails to be detected as the original object by YOLOv8.

In [ ]:
import numpy as np
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 1. Load the Model
model = YOLO("YOLO11n-cls.pt")

# 2. Load the Reference Image
original_image = Image.open("nika.png").convert("RGB")
original_array = np.array(original_image, dtype=np.float32)

print(f"Image Loaded. Shape: {original_array.shape}")
plt.imshow(original_image)
plt.axis('off')
plt.show()

In [ ]:
# 3. Check Baseline Detection (Sanity Check)
results = model(original_image)
for r in results:
    for box in r.boxes:
        print(f"Detected Class: {int(box.cls[0])} - Conf: {float(box.conf[0]):.2f}")

## Your Attack Code
Implement your attack below. You likely want to use a gradient-based method (like FGSM).

In [ ]:
adversarial_image = original_image.copy()


In [ ]:
adv_array = np.array(adversarial_image, dtype=np.float32)
diff = adv_array - original_array
l2_dist = np.linalg.norm(diff)

print(f"Current L2 Distance: {l2_dist:.2f}")

if l2_dist > 50:
    print("⚠️ WARNING: L2 Distance is too high!")
else:
    print("✅ L2 Distance is within limits.")

In [ ]:
import requests
import io, os
import json

# This is within the same k8 cluster, so you can just run this.
API_URL = os.environ.get("VERIFICATION_SERVER")
def submit_solution(image_obj):
    # Convert PIL Image to Bytes
    buf = io.BytesIO()
    image_obj.save(buf, format="JPEG")
    buf.seek(0)
    
    # Send POST request to the internal server
    try:
        response = requests.post(API_URL, files={'image': buf}, timeout=5)
        data = response.json()
        
        if data.get("success"):
            print("🎉 VICTORY!")
            print(f"Flag: {data['flag']}")
            print(f"Final L2 Distance: {data['l2_distance']:.2f}")
        else:
            print("❌ Attempt Failed")
            print(f"Message: {data['message']}")
            if 'l2_distance' in data:
                print(f"L2 Distance: {data['l2_distance']:.2f}")
                
    except requests.exceptions.ConnectionError:
        print(f"⚠️ Error: Could not connect to {API_URL}. Is the challenge server running?")
    except Exception as e:
        print(f"⚠️ Error: {e}")

# Call the function with your adversarial image
submit_solution(adversarial_image)